# Scanned PDF — Multi Pass Extraction

Extract structured data from a **scanned** (image-based) PDF using multi-pass extraction.

Combines the scanned strategy (vision LLM parsing) with multi-pass extraction (per-page LLM calls merged into a single result). Best for long scanned documents.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
import sys

from pydantic import BaseModel

from doc_intelligence import PDFExtractionMode, PDFProcessor
from doc_intelligence.pdf.types import ParseStrategy

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from utils import show_pdf_with_bboxes

PDF_PATH = "/Users/zeel/Public/ms/open_source/document_ai/data/sample_invoice.pdf"
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output")
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
class Output(BaseModel):
    blue_shield_charge: float

In [4]:
processor = PDFProcessor(
    provider="openai",
    model="gpt-5.4",
    include_citations=True,
    extraction_mode=PDFExtractionMode.MULTI_PASS,
    strategy=ParseStrategy.SCANNED,
)

In [5]:
response = processor.extract(PDF_PATH, Output, page_numbers=[0, 1])

2026-04-06 23:49:42.602 | DEBUG    | doc_intelligence.pdf.parser:_parse_scanned_vlm:231 - VLM parse: batch starting at page 0, 1 page(s)
2026-04-06 23:49:42.602 | DEBUG    | doc_intelligence.llm:generate:29 - OpenAILLM: generate: 1 image(s), model=gpt-5.4
2026-04-06 23:49:53.110 | INFO     | doc_intelligence.pdf.processor:extract:71 - Document parsed successfully
2026-04-06 23:49:53.110 | INFO     | doc_intelligence.pdf.formatter:format_document_for_llm:102 - Formatting 1 pages
2026-04-06 23:49:53.110 | DEBUG    | doc_intelligence.llm:generate:41 - OpenAILLM: generate: model=gpt-5.4
2026-04-06 23:49:55.005 | DEBUG    | doc_intelligence.pdf.extractor:_run_multi_pass:157 - PDFExtractor: multi-pass: pass1 complete: blue_shield_charge=300.0
2026-04-06 23:49:55.007 | INFO     | doc_intelligence.pdf.formatter:format_document_for_llm:102 - Formatting 1 pages
2026-04-06 23:49:55.009 | DEBUG    | doc_intelligence.llm:generate:41 - OpenAILLM: generate: model=gpt-5.4
2026-04-06 23:49:56.845 | DEB

In [6]:
response

ExtractionResult(data=Output(blue_shield_charge=300.0), metadata={'blue_shield_charge': {'value': 300.0, 'citations': [{'page': 0, 'bboxes': [{'x0': 0.017663817663817662, 'top': 0.0, 'x1': 0.9635327635327635, 'bottom': 0.917741935483871}]}]}})

In [7]:
assert response.metadata is not None
show_pdf_with_bboxes(
    PDF_PATH,
    response.metadata["blue_shield_charge"]["citations"][0],
    out_file=os.path.join(OUT_DIR, "scanned_multi_pass_bbox.png"),
)

  -> saved to /Users/zeel/Public/ms/open_source/document_ai/notebooks/pdf/output/scanned_multi_pass_bbox.png


In [8]:
response

ExtractionResult(data=Output(blue_shield_charge=300.0), metadata={'blue_shield_charge': {'value': 300.0, 'citations': [{'page': 0, 'bboxes': [{'x0': 0.017663817663817662, 'top': 0.0, 'x1': 0.9635327635327635, 'bottom': 0.917741935483871}]}]}})